In [1]:
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt

In [91]:
def resize(image, maximum_s = 700, devided_by = 3):
    h, w = image.shape[0], image.shape[1]
    if h > maximum_s or w > maximum_s:
        if h > w*1.5:
            image = cv.resize(image, (w // devided_by, h // (devided_by+1)), interpolation=cv.INTER_AREA)
        else:
            image = cv.resize(image, (w // devided_by, h // devided_by), interpolation=cv.INTER_AREA)
    return image



def mse(img1, img2):
    error = np.sum((img1.astype('float') - img2.astype('float')) ** 2)
    error /= float(img1.shape[0] * img1.shape[1])

    return error
    

In [167]:
cap = cv.VideoCapture("data\street1.mp4")
vehicles = 0
frame_ignore = 0
car = False

while True:
    rec, f_jump = cap.read()
    rec, f_jump = cap.read()
    rec, f_jump = cap.read()

    rec, frame1 = cap.read()
    rec, frame2 = cap.read()
    if not rec:
        break

    if frame_ignore == 1:
        frame_ignore = 0
        continue
    else:
        frame_ignore += 1

    frame1 = resize(frame1)
    frame2 = resize(frame2)

    focus1 = frame1[720:955, 0:715].copy()
    focus2 = frame2[720:955, 0:715].copy()


    frame_diff = cv.absdiff(focus1, focus2)

    gray = cv.cvtColor(frame_diff, cv.COLOR_BGR2GRAY)
    _ , thresh = cv.threshold(gray, 10, 255, cv.THRESH_BINARY)
    dilated = cv.dilate(thresh, (13,13), iterations=1)
    closed = cv.morphologyEx(dilated, cv.MORPH_CLOSE, (9,9))
    blured = cv.GaussianBlur(closed, (9,9), 1)
    eroded = cv.erode(blured, (10,10), iterations=1)

    contours, _ = cv.findContours(eroded, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    
    for c in contours:
        if cv.contourArea(c) > 1300:
            x,y,w,h = cv.boundingRect(c)
            cv.rectangle(focus2, (x,y), (x+w,y+h), (0,0,255), 3)
            

    section = eroded[0:40, 0:715].copy()
    check_section_contours, _ = cv.findContours(section, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    sorted_contours = sorted(check_section_contours, key=cv.contourArea, reverse=True)[:1]

    for c in sorted_contours:
        if cv.contourArea(c) > 1300:
            x,y,w,h = cv.boundingRect(c)
            print(x,y,w,h)

            if w > 100 and car == True:
                continue
            elif w > 100 and car == False:
                vehicles += 1
                car = True
            else:
                vehicles += 1
            

                
                

    frame2[720:955, 0:715] = focus2



    cv.rectangle(frame2, (0,720), (715,955), (0,0,255), 3)
    cv.line(frame2, (0,760), (715,760), (0,255,0), 3)
    cv.putText(frame2, f"vehicles: {vehicles}", (10,30), cv.FONT_HERSHEY_COMPLEX, 1, (0,0,255), 1)


    cv.imshow("blublured",blured)
    cv.imshow("dilated", dilated)
    cv.imshow("eroded", eroded)
    cv.imshow("frame", frame2)
    if cv.waitKey(1) & 0xFF == 27:
        break


cv.destroyAllWindows()
cap.release()

324 0 97 40
337 0 89 35
423 0 58 40
576 7 62 33
568 0 51 40
397 12 78 28
397 0 63 40
261 0 58 40
145 0 59 40
411 0 58 40
508 0 144 40
510 0 128 40
500 0 118 40
592 0 72 40
593 0 57 40
293 0 59 40
198 0 67 40
224 0 57 40
299 0 55 40
19 0 116 40
621 0 53 40
301 0 128 40
315 0 112 40
328 0 100 40
613 0 102 40
597 0 118 40
270 0 60 40
282 0 90 40
313 0 71 40
557 3 66 37
552 0 60 40
